In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              GradientBoostingClassifier, GradientBoostingRegressor)
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, mean_absolute_error, mean_squared_error, r2_score)
from swiftlog_config import FEATURES
from swiftlog_utils import load_problem, make_preprocessor

In [4]:
def models_for(task):
    if task == "classification":
        return {
            "Baseline (majority)": DummyClassifier(strategy="most_frequent"),
            "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
            "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=20,
                                class_weight="balanced", random_state=42, n_jobs=-1),
            "Gradient Boosting": GradientBoostingClassifier(n_estimators=150, max_depth=3,
                                learning_rate=0.05, random_state=42),
        }
    return {
        "Baseline (mean)": DummyRegressor(strategy="mean"),
        "Ridge": Ridge(alpha=10.0),
        "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=10,
                                random_state=42, n_jobs=-1),
        "Gradient Boosting": GradientBoostingRegressor(n_estimators=150, max_depth=3,
                                learning_rate=0.05, random_state=42),
    }

def evaluate(task, y, pred, proba=None):
    if task == "classification":
        return dict(Accuracy=accuracy_score(y, pred),
                    Precision=precision_score(y, pred, zero_division=0),
                    Recall=recall_score(y, pred), F1=f1_score(y, pred),
                    ROC_AUC=roc_auc_score(y, proba))
    mse = mean_squared_error(y, pred)
    return dict(MAE=mean_absolute_error(y, pred), MSE=mse,
                RMSE=np.sqrt(mse), R2=r2_score(y, pred))

rows = []
for name in FEATURES:
    cfg, sets = load_problem(name)
    Xtr, ytr = sets["train"]
    Xva, yva = sets["val"]
    for mname, est in models_for(cfg["task"]).items():
        pipe = Pipeline([("prep", make_preprocessor(cfg)), ("model", est)]).fit(Xtr, ytr)
        pred = pipe.predict(Xva)
        proba = pipe.predict_proba(Xva)[:, 1] if cfg["task"] == "classification" else None
        rows.append({"Problem": name, "Model": mname, **evaluate(cfg["task"], yva, pred, proba)})

res = pd.DataFrame(rows).round(3)
for name, g in res.groupby("Problem", sort=False):
    print("\n", name.upper())
    print(g.drop(columns="Problem").dropna(axis=1, how="all").to_string(index=False))

res.to_csv("../Models/validation_results_round1.csv", index=False)


 DELAY
              Model  Accuracy  Precision  Recall    F1  ROC_AUC
Baseline (majority)     0.648      0.648   1.000 0.786    0.500
Logistic Regression     0.488      0.643   0.471 0.544    0.505
      Random Forest     0.495      0.636   0.514 0.569    0.496
  Gradient Boosting     0.641      0.654   0.947 0.774    0.530

 DEMAND
            Model    MAE     MSE   RMSE     R2
  Baseline (mean) 10.688 171.378 13.091 -0.123
            Ridge 10.667 172.053 13.117 -0.127
    Random Forest 10.809 177.107 13.308 -0.160
Gradient Boosting 10.885 182.512 13.510 -0.196

 CUSTOMER
            Model      MAE          MSE     RMSE     R2
  Baseline (mean) 5019.246 38186705.514 6179.539 -0.003
            Ridge 4964.154 37609594.362 6132.666  0.012
    Random Forest 5102.009 38780060.207 6227.364 -0.019
Gradient Boosting 4996.058 38954344.566 6241.342 -0.024

 WAREHOUSE
            Model   MAE   MSE  RMSE     R2
  Baseline (mean) 0.065 0.009 0.095 -0.001
            Ridge 0.065 0.009 0.095  0.